# Airflow 2: Publish an asset event

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/airflow/airflow-02-assets.ipynb)

Run a producer DAG and inspect the update it records for an asset-scheduled downstream DAG.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q apache-airflow~=3.4

## Setup and sample data

Create a local Airflow metadata database and DAG folder.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory()
home = Path(workspace.name)
dags = home / "dags"
dags.mkdir()
env = {**os.environ, "AIRFLOW_HOME": str(home), "AIRFLOW__CORE__LOAD_EXAMPLES": "False"}


def run_local(*args):
    run = subprocess.run([sys.executable, *args], env=env, capture_output=True, text=True)
    if run.returncode:
        raise RuntimeError(run.stderr or run.stdout)
    for line in run.stdout.splitlines():
        if line.startswith("LAB "):
            print(line)


run_local("-m", "airflow", "db", "migrate")

## Emit a data update on successful publication

An asset represents a logical dataset. A producer task declares it as an outlet. An asset-scheduled consumer declares that same asset in its schedule.

**Where it matters**

Trigger downstream work when a curated dataset is ready.

**Watch out for**

The lab runs the producer with dag.test and reads its recorded asset event. A deployed scheduler handles the consumer trigger. An asset event signals success and does not prove a dataset meets its quality contract.

In [ ]:
file = dags / "assets_lab.py"
file.write_text("""
from datetime import datetime, timezone
from airflow.sdk import Asset, dag, task

orders = Asset("orders_curated")
start = datetime(2026, 1, 1, tzinfo=timezone.utc)

@dag(schedule=None, start_date=start)
def build_orders():
    @task(outlets=[orders])
    def publish():
        print("LAB published orders_curated")

    publish()

@dag(schedule=[orders], start_date=start)
def summarize_orders():
    @task
    def summarize():
        print("LAB reading updated orders")

    summarize()

producer = build_orders()
consumer = summarize_orders()

if __name__ == "__main__":
    from airflow.models.asset import AssetEvent
    from airflow.utils.session import create_session

    print("LAB producer state:", producer.test().state)
    with create_session() as session:
        print("LAB recorded asset events:", session.query(AssetEvent).count())
""")
run_local(str(file))

**What to notice:** The producer succeeds and one asset event is stored in the temporary metadata database.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Should a producer emit the asset event before its table commit?

In [ ]:
# Try your solution here.

In [ ]:
print("Publish successfully first so consumers can safely read the table.")

## Key takeaways

- Trigger downstream work when a curated dataset is ready.

**Next:** [Airflow 3: Backfill data intervals](airflow-03-backfill.ipynb)